In [1]:
from pprint import pprint
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [2]:
@tool
def search_topic_notes(topic: str) -> str:
    """Search a small internal knowledge base and return research notes for a topic."""
    kb = {
        "retrieval augmented generation": (
            "- RAG combines retrieval (searching external knowledge) with generation.\n"
            "- It helps reduce hallucinations by grounding answers in retrieved documents.\n"
            "- Typical pipeline: embed query -> retrieve docs -> pass docs + question to LLM.\n"
            "- Quality depends on chunking, embeddings, retrieval strategy, and prompting.\n"
            "- Common use cases: enterprise Q&A, support bots, internal knowledge assistants."
        ),
        "vector database": (
            "- Vector databases store embeddings for semantic search.\n"
            "- They enable nearest-neighbor retrieval by meaning, not exact keywords.\n"
            "- Common concerns: latency, indexing, filtering, metadata, hybrid search.\n"
            "- Often used in RAG systems to retrieve relevant chunks."
        ),
        "agentic ai": (
            "- Agentic AI refers to systems that can plan, use tools, and take actions.\n"
            "- Agents are useful for multi-step workflows, not just single responses.\n"
            "- Good design needs clear tool boundaries, observability, and error handling.\n"
            "- Overusing agents can increase latency and cost."
        ),
    }

    topic_lower = topic.lower().strip()

    # direct match first
    if topic_lower in kb:
        return kb[topic_lower]

    # simple fuzzy contains match
    for k, v in kb.items():
        if topic_lower in k or k in topic_lower:
            return v

    return (
        f"No exact notes found for '{topic}'.\n"
        "- Try a more specific topic.\n"
        "- If this were production, this tool would call web search / MCP fetch / vector DB."
    )

In [3]:
MODEL_NAME = "openai:gpt-5-nano"

research_agent = create_agent(
    model=MODEL_NAME,
    tools=[search_topic_notes],
    system_prompt=(
        "You are a research analyst. "
        "Use your tool to gather notes. "
        "Return concise, factual bullet points and include caveats when data is limited."
    ),
)

In [4]:
writer_agent = create_agent(
    model=MODEL_NAME,
    tools=[],
    system_prompt=(
        "You are a professional writer. "
        "Turn research notes into clear, engaging writing. "
        "Do not invent facts not present in the notes."
    ),
)

In [5]:
@tool
def call_research_agent(topic: str) -> str:
    """Call the research subagent to gather factual notes about a topic."""
    response = research_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        f"Research this topic and return bullet-point notes only: {topic}"
                    )
                )
            ]
        }
    )
    return response["messages"][-1].content

In [6]:
@tool
def call_writer_agent(topic: str, research_notes: str, audience: str = "general") -> str:
    """Call the writer subagent to create polished content from research notes."""
    response = writer_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        f"Topic: {topic}\n"
                        f"Audience: {audience}\n\n"
                        f"Research notes:\n{research_notes}\n\n"
                        "Write a short LinkedIn-style post (5-8 lines) with a clear takeaway."
                    )
                )
            ]
        }
    )
    return response["messages"][-1].content

In [7]:
SYSTEM_PROMPT = (
    "You are an editor/orchestrator. "
    "When the user asks for written content on a topic, first call the research subagent, "
    "then call the writer subagent using the research notes. "
    "Return only the final polished output unless the user asks for the notes."
)

main_agent = create_agent(
    model=MODEL_NAME,
    tools=[call_research_agent, call_writer_agent],
    system_prompt=SYSTEM_PROMPT,
)

In [8]:
question = HumanMessage(
    content=(
        "Create a short LinkedIn post for a technical audience about Retrieval Augmented Generation (RAG). "
        "Research first, then write."
    )
)

response = main_agent.invoke({"messages": [question]})

pprint(response)
print("\nFINAL ANSWER:\n")
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='Create a short LinkedIn post for a technical audience about Retrieval Augmented Generation (RAG). Research first, then write.', additional_kwargs={}, response_metadata={}, id='15a782ea-99ee-4930-bb5b-4f928ecff6b1'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 545, 'prompt_tokens': 257, 'total_tokens': 802, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DFWeFBmMfX1k3uUi9E4tcKM99YsoK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb6c3-7ec3-7fe1-adce-bc300b54db5b-0', tool_calls=[{'name': 'call_research_agent', 'args': {'topic': 'Retrieval Augmented Generat

In [15]:
question = HumanMessage(
    content=(
        "Create a short LinkedIn post for a technical audience about Agentic AI. "
        "Research first, then write."
    )
)

response = main_agent.invoke({"messages": [question]})

print("\nFINAL ANSWER:\n")
print(response["messages"][-1].content)


FINAL ANSWER:

Agentic AI refers to systems that can plan, use tools, and take actions. These agents shine in multi-step workflows, not just one-off responses. Design well: define clear tool boundaries, ensure observability, and implement robust error handling. Be mindful—overusing agents can increase latency and cost. Definitions vary across literature; our summary reflects a small internal knowledge base. Takeaway: use agentic AI where it truly adds value, and pair it with disciplined design to keep costs in check.

#AgenticAI #AI #AIEngineering #Automation #ML #Technology


In [14]:
question = HumanMessage(
    content=(
        "Create a short LinkedIn post for a technical audience about vector database. "
        "Research first, then write."
    )
)

response = main_agent.invoke({"messages": [question]})

print("\nFINAL ANSWER:\n")
print(response["messages"][-1].content)


FINAL ANSWER:

Vector databases store embeddings to enable semantic search—not just keyword matching. They enable nearest-neighbor retrieval by meaning, so you surface related ideas even when phrasing differs.

Common concerns: latency, indexing strategy, filtering, metadata handling, and hybrid search (text + vector).

They’re a core piece in retrieval-augmented generation (RAG) to fetch relevant chunks or documents.

Caveat: performance and features vary by product—index types (HNSW, IVF), hardware acceleration, embedding dimensionality, dataset size, and metadata support.

Takeaway: if you’re building AI-powered search or RAG workflows, prototype across your data and prioritize the features that matter to your use case. Would love to hear your experiences with vector databases in practice.


In [ ]:
"""
Your architecture has two knowledge sources:
Tool knowledge (your mock KB)
Model knowledge (LLM pretrained knowledge)

If you don’t explicitly restrict the model, it may use #2.
"""

question = HumanMessage(
    content=(
        "Create a short LinkedIn post for a technical audience about Super Intelligence. "
        "Research first, then write."
    )
)

response = main_agent.invoke({"messages": [question]})

print("\nFINAL ANSWER:\n")
print(response["messages"][-1].content)


FINAL ANSWER:

Super intelligence: an intellect that vastly surpasses the best human minds across science, problem solving, and social reasoning.

Key distinctions: Narrow AI (specialized tasks), AGI (human-level across tasks), and ASI (far beyond human in nearly all domains).

Core idea: an intelligence explosion via recursive self-improvement could outpace human control and understanding.

Why it matters: alignment and safety—ensuring ASI’s goals stay aligned with human well-being as capabilities grow.

Safety research focus: value alignment, corrigibility, interpretability, robustness, safe exploration, and governance mechanisms (kill switches, containment, international coordination).

Timelines and pathways are highly uncertain—takeoffs could be slow or fast, and breakthroughs could come from hardware/software shifts, self-improving architectures, or hybrid symbolic-ML approaches.

Takeaways for builders: embed safety by design, pursue transparent evaluation, and advocate for cro